In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Läs in data
df = pd.read_csv("historical_data.csv")  

target_name = "is_suspicious"

X = df.drop(columns=[target_name, "id"])
y = df[target_name]

numeric_features = [
    "day",
    "account_age_days",
    "num_prev_listings",
    "prev_reports_30d",
    "verification_level",
    "price",
    "num_images",
    "message_length",
    "contains_off_platform",
    "urgency_words",
    "payment_attempt",
    "time_to_first_response_min"
]

categorical_features = [
    "event_type",
    "category",
    "region",
    "device"
]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Setup klar")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

Setup klar
X_train: (8400, 16)
X_test: (3600, 16)


## 4. Hyperparameter-tuning av vald modell

Vi valde att optimera den modell som gick vidare från modelljämförelsen. För att hålla lösningen enkel testade vi ett mindre antal hyperparametrar med GridSearchCV och 5-fold stratified cross-validation. Eftersom vårt kravkort fokuserar på att minska onödiga flaggningar valde vi att optimera mot precision.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

param_grid = {
    "model__C": [0.1, 1, 10],
    "model__class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="precision",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Bästa parametrar:", grid_search.best_params_)
print("Bästa precision (CV):", round(grid_search.best_score_, 3))

best_model = grid_search.best_estimator_

Bästa parametrar: {'model__C': 10, 'model__class_weight': None}
Bästa precision (CV): 0.595


In [3]:
results = pd.DataFrame(grid_search.cv_results_)
results = results[["params", "mean_test_score", "std_test_score"]] \
    .sort_values("mean_test_score", ascending=False)

results.head(5)

,params,mean_test_score,std_test_score
4,"{'model__C': 10, 'model__class_weight': None}",0.595455,0.113113
2,"{'model__C': 1, 'model__class_weight': None}",0.590455,0.112245
0,"{'model__C': 0.1, 'model__class_weight': None}",0.590256,0.109873
1,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.192561,0.012049
3,"{'model__C': 1, 'model__class_weight': 'balanc...",0.191570,0.012161


In [4]:
test_proba = best_model.predict_proba(X_test)[:, 1]
print("Antal sannolikheter för testdata:", len(test_proba))

Antal sannolikheter för testdata: 3600


### Resultat från tuning

Grid search visade att den bästa varianten av Logistic Regression fick parametrarna **C = 10** och **class_weight = None**. Den bästa genomsnittliga precisionen i cross-validation blev **0.595**. Detta tyder på att en försiktig modell utan balanserade klassvikter passar vårt kravkort bättre, eftersom målet är att minska onödiga flaggningar.

# 5. Test Different Thresholds


In [5]:
from sklearn.metrics import confusion_matrix, precision_score

# Vi testar olika thresholds (gränsvärden) från standard 50% upp till 95%
thresholds = [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95]

print("Threshold | Totalt Flaggade | Oskyldiga (False Positives) | Precision (Hur säkra vi är)")
print("-" * 85)

for t in thresholds:
    # Skapa nya prediktioner baserat på nuvarande threshold
    y_pred_custom = (test_proba >= t).astype(int)
    
    # Räkna ut hur många som blev rätt och fel
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_custom).ravel()
    
    total_flagged = tp + fp
    precision = precision_score(y_test, y_pred_custom, zero_division=0)
    
    print(f"{t:<9} | {total_flagged:<15} | {fp:<29} | {precision:.1%}")

Threshold | Totalt Flaggade | Oskyldiga (False Positives) | Precision (Hur säkra vi är)
-------------------------------------------------------------------------------------
0.5       | 23              | 11                            | 52.2%
0.6       | 12              | 5                             | 58.3%
0.7       | 2               | 1                             | 50.0%
0.8       | 0               | 0                             | 0.0%
0.85      | 0               | 0                             | 0.0%
0.9       | 0               | 0                             | 0.0%
0.95      | 0               | 0                             | 0.0%


**Analys av tröskelvärden:**

Som vi ser i tabellen ovan är standardvärdet 0.5 (50%) problematiskt för Lina. Vid 0.5 flaggar vi 23 personer, men 11 av dem är helt oskyldiga (False Positives). Det är en felmarginal på nästan 50%, vilket skulle skapa ett enormt tryck på kundtjänst.

Notera även att vid 0.8 och uppåt flaggas 0 personer. Det betyder att modellens högsta sannolikhet för testdatat ligger någonstans i 70-procentsspannet. Vi kan alltså inte bara "höja ribban" till 90% för att vara säkra, då skulle vi inte fånga några skurkar alls.

### Option 1: The Micro-Threshold

In [6]:
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score

# Skapar en lista med värden: 0.61, 0.62, ..., 0.69
micro_thresholds = np.arange(0.61, 0.70, 0.01)

print("Threshold | Totalt Flaggade | Oskyldiga (FP) | Precision")
print("-" * 65)

for t in micro_thresholds:
    y_pred_custom = (test_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_custom).ravel()
    
    total_flagged = tp + fp
    precision = precision_score(y_test, y_pred_custom, zero_division=0)
    
    print(f"{t:<9.2f} | {total_flagged:<15} | {fp:<14} | {precision:.1%}")

Threshold | Totalt Flaggade | Oskyldiga (FP) | Precision
-----------------------------------------------------------------
0.61      | 12              | 5              | 58.3%
0.62      | 11              | 4              | 63.6%
0.63      | 9               | 4              | 55.6%
0.64      | 9               | 4              | 55.6%
0.65      | 7               | 3              | 57.1%
0.66      | 6               | 3              | 50.0%
0.67      | 5               | 3              | 40.0%
0.68      | 4               | 2              | 50.0%
0.69      | 3               | 2              | 33.3%


**Strategi 1: Optimering av tröskelvärde**

Genom att titta närmare på spannet 0.60–0.69 ser vi hur precisionen förändras för varje procentenhet.

Slutsats: Genom att ligga runt 0.65 kan vi halvera antalet oskyldiga som drabbas jämfört med standardinställningen, samtidigt som vi behåller en rimlig volym för granskning. Detta balanserar Linas krav på att "oskyldiga inte ska bli arga".

### Option 2: Top-10 Prioritization Rule


In [7]:
# Skapa en tabell som kopplar ihop facit med modellens sannolikhet
results_df = pd.DataFrame({'Target': y_test, 'Probability': test_proba})

# Sortera så de mest misstänkta hamnar högst upp
results_df = results_df.sort_values(by='Probability', ascending=False)

# Välj ut top 10
top_x = 10
top_x_df = results_df.head(top_x)

fp_in_top_x = len(top_x_df[(top_x_df['Target'] == 0)])
tp_in_top_x = len(top_x_df[(top_x_df['Target'] == 1)])
precision_top_x = tp_in_top_x / top_x if top_x > 0 else 0

print(f"Bland de top {top_x} mest misstänkta fallen:")
print(f"Sanna skurkar (TP): {tp_in_top_x}")
print(f"Oskyldiga (False Positives): {fp_in_top_x}")
print(f"Precision: {precision_top_x:.1%}")

Bland de top 10 mest misstänkta fallen:
Sanna skurkar (TP): 6
Oskyldiga (False Positives): 4
Precision: 60.0%


**Strategi 2: Topp-lista för granskning**

Istället för ett fast gränsvärde kan vi leverera en daglig "Topp 10"-lista till Lina.

Fördel: Detta ger kundtjänst en förutsägbar arbetsbörda. Vi fokuserar resurserna där sannolikheten för bedrägeri är som högst. Som visas i resultatet ovan är precisionen högre i den absoluta toppen av listan.

### Find the Innocent User Example

In [8]:
# Sätt gränsvärdet vi vill undersöka (Vi använder 0,60 som tröskelvärde i detta exempel eftersom vi vet att det flaggar 5 oskyldiga personer)
vald_threshold = 0.60

# Hitta de som faktiskt är oskyldiga (Facit/Target = 0)
oskyldiga_mask = (y_test == 0)

# Hitta de som modellen trodde var skurkar (Sannolikhet >= 0.60)
flaggade_mask = (test_proba >= vald_threshold)

# Plocka ut dessa rader från testdatan
falska_positiva = X_test[oskyldiga_mask & flaggade_mask]

print(f"Antal oskyldiga användare som flaggades vid gränsvärdet {vald_threshold}: {len(falska_positiva)}")
print("-" * 80)
print("Här är ett konkret exempel på en oskyldig användare som modellen flaggade av misstag:")

# Visa den allra första oskyldiga användaren i en tabell
display(falska_positiva.head(1))

Antal oskyldiga användare som flaggades vid gränsvärdet 0.6: 5
--------------------------------------------------------------------------------
Här är ett konkret exempel på en oskyldig användare som modellen flaggade av misstag:


,day,event_type,category,region,device,account_age_days,num_prev_listings,prev_reports_30d,verification_level,price,num_images,message_length,contains_off_platform,urgency_words,payment_attempt,time_to_first_response_min
7866,7,message_send,furniture,NaN,android,17.0,2,1,0,184.36,1,100,1,0,0,8.2


**Varför flaggas oskyldiga? (Case-studie för Lina)**

Här har vi extraherat ett konkret exempel på en användare som är oskyldig (Target=0) men som modellen flaggade (Sannolikhet > 0.60).

Om vi tittar på raden ovan kan vi se att användaren kanske har ett nystartat konto eller använde ord som "bråttom". Detta visar varför vi inte kan ha en policy som automatiskt stänger av konton.

**Rekommenderad Policy:** > Istället för permanent avstängning föreslår vi "mjuk friktion". När en användare flaggas pausas annonsen temporärt och användaren ombeds verifiera sig (t.ex. via SMS). Detta stoppar automatiserade bedragare men låter oskyldiga användare som denna fortsätta utan att behöva kontakta kundtjänst.